In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl

REPO_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "common" / "paths.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.paths import ROOT

from common.dataset import make_loader
from common.metrics import calculate_fid, dump_json, image_metric_frame, load_weight_only, regression_frames, save_translation_grid, save_weight_only, summarize_image_metrics, summarize_regression
from common.models import EmeanRegressor, PatchDiscriminator, TranslationNetwork, initialize_map2sat
from common.paths import PRETRAINED_GENERATOR
from common.train import BestStateCallback, EpochRecorder

DATASET_NAME = "pancreas"
EXPERIMENT_NAME = "all_innovations"
NOTEBOOK_DIR = ROOT / "experiment" / DATASET_NAME / "02_ablation"
PAIR_CSV = ROOT / "experiment" / DATASET_NAME / "01_data_processing" / "cache" / "pairs_seed_42.csv"
EXTERNAL_CSV = ROOT / "experiment" / DATASET_NAME / "01_data_processing" / "cache" / "external_pairs.csv"
training = False
USE_FEM = True
USE_PRETRAINED = True
USE_EMEAN = True
EXPORT_ALL_SPLITS = True
SEED = 42
BATCH_SIZE_GENERATOR = 8
BATCH_SIZE_EMEAN = 16
GENERATOR_EPOCHS = 80
EMEAN_EPOCHS = 60
NUM_WORKERS = 4
LEARNING_RATE_GENERATOR = 2e-4
LEARNING_RATE_EMEAN = 1e-3
GENERATOR_WEIGHT = NOTEBOOK_DIR / "all_innovations_generator.pt"
EMEAN_WEIGHT = NOTEBOOK_DIR / "all_innovations_emean_head.pt"
METRICS_PATH = NOTEBOOK_DIR / "metrics_all_innovations.json"

pl.seed_everything(SEED, workers=True)
if not PAIR_CSV.is_file():
    raise FileNotFoundError(PAIR_CSV)
frame = pd.read_csv(PAIR_CSV)
train_frame = frame.loc[frame["split"] == "train"].copy()
validation_frame = frame.loc[frame["split"] == "val"].copy()
test_frame = frame.loc[frame["split"] == "test"].copy()
train_loader = make_loader(train_frame, BATCH_SIZE_GENERATOR, True, True, NUM_WORKERS)
validation_loader = make_loader(validation_frame, BATCH_SIZE_GENERATOR, False, False, NUM_WORKERS)
test_loader = make_loader(test_frame, BATCH_SIZE_GENERATOR, False, False, NUM_WORKERS)
train_emean_loader = make_loader(train_frame, BATCH_SIZE_EMEAN, True, True, NUM_WORKERS)
train_evaluation_loader = make_loader(train_frame, BATCH_SIZE_EMEAN, False, False, NUM_WORKERS)
validation_evaluation_loader = make_loader(validation_frame, BATCH_SIZE_EMEAN, False, False, NUM_WORKERS)
test_evaluation_loader = make_loader(test_frame, BATCH_SIZE_EMEAN, False, False, NUM_WORKERS)
external_frame = pd.read_csv(EXTERNAL_CSV) if EXTERNAL_CSV.is_file() else None
external_loader = make_loader(external_frame, BATCH_SIZE_GENERATOR, False, False, NUM_WORKERS) if external_frame is not None else None
external_evaluation_loader = make_loader(external_frame, BATCH_SIZE_EMEAN, False, False, NUM_WORKERS) if external_frame is not None else None

In [ ]:
class TranslationLightning(pl.LightningModule):
    def __init__(self, use_fem, learning_rate=2e-4, reconstruction_weight=100.0):
        super().__init__()
        self.automatic_optimization = False
        self.network = TranslationNetwork(use_fem=use_fem)
        self.discriminator = PatchDiscriminator()
        self.learning_rate = learning_rate
        self.reconstruction_weight = reconstruction_weight
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.reconstruction_loss = nn.L1Loss()

    def forward(self, inputs):
        return self.network(inputs)

    def training_step(self, batch, batch_index):
        generator_optimizer, discriminator_optimizer = self.optimizers()
        gray = batch["gray"]
        target = batch["swe"]
        generated = self(gray)
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(True)
        real_logits = self.discriminator(gray, target)
        fake_logits = self.discriminator(gray, generated.detach())
        discriminator_loss = 0.5 * (
            self.adversarial_loss(real_logits, torch.ones_like(real_logits))
            + self.adversarial_loss(fake_logits, torch.zeros_like(fake_logits))
        )
        discriminator_optimizer.zero_grad()
        self.manual_backward(discriminator_loss)
        discriminator_optimizer.step()
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(False)
        fake_logits = self.discriminator(gray, generated)
        adversarial = self.adversarial_loss(fake_logits, torch.ones_like(fake_logits))
        reconstruction = self.reconstruction_loss(generated, target)
        generator_loss = adversarial + self.reconstruction_weight * reconstruction
        generator_optimizer.zero_grad()
        self.manual_backward(generator_loss)
        generator_optimizer.step()
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(True)
        self.log("train_generator_loss", generator_loss, prog_bar=True)
        self.log("train_discriminator_loss", discriminator_loss, prog_bar=True)
        self.log("train_reconstruction", reconstruction)
        return generator_loss

    def validation_step(self, batch, batch_index):
        generated = self(batch["gray"])
        loss = self.reconstruction_loss(generated, batch["swe"])
        self.log("val_reconstruction", loss, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_index):
        generated = self(batch["gray"])
        loss = self.reconstruction_loss(generated, batch["swe"])
        self.log("test_reconstruction", loss, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_index, dataloader_index=0):
        return {
            "gray": batch["gray"].detach().cpu(),
            "prediction": self(batch["gray"]).detach().cpu(),
            "target": batch["swe"].detach().cpu(),
            "emean": batch["emean"].detach().cpu(),
            "image_name": list(batch["image_name"]),
            "patient_id": list(batch["patient_id"]),
        }

    def configure_optimizers(self):
        generator_optimizer = torch.optim.Adam(
            self.network.parameters(),
            lr=self.learning_rate,
            betas=(0.5, 0.999),
        )
        discriminator_optimizer = torch.optim.Adam(
            self.discriminator.parameters(),
            lr=self.learning_rate,
            betas=(0.5, 0.999),
        )
        return [generator_optimizer, discriminator_optimizer]

In [ ]:
class EmeanRegressionLightning(pl.LightningModule):
    def __init__(
        self,
        input_mode,
        target_mean,
        target_std,
        translation=None,
        learning_rate=1e-3,
    ):
        super().__init__()
        self.input_mode = input_mode
        self.translation = translation
        if self.translation is not None:
            self.translation.eval()
            for parameter in self.translation.parameters():
                parameter.requires_grad_(False)
        self.regressor = EmeanRegressor()
        self.register_buffer("target_mean", torch.tensor(float(target_mean)))
        self.register_buffer("target_std", torch.tensor(float(target_std)))
        self.learning_rate = learning_rate
        self.loss_function = nn.HuberLoss(delta=1.0)

    def on_train_epoch_start(self):
        if self.translation is not None:
            self.translation.eval()

    def _images(self, batch):
        if self.input_mode == "gray":
            return batch["gray"]
        if self.input_mode == "real_swe":
            return batch["swe"]
        if self.input_mode == "virtual_swe":
            with torch.no_grad():
                return self.translation(batch["gray"])
        raise ValueError(self.input_mode)

    def _standardized_prediction(self, batch):
        return self.regressor(self._images(batch))

    def forward(self, batch):
        standardized = self._standardized_prediction(batch)
        return torch.expm1(standardized * self.target_std + self.target_mean).clamp_min(0.0)

    def _loss(self, batch):
        target = (torch.log1p(batch["emean"]) - self.target_mean) / self.target_std
        return self.loss_function(self._standardized_prediction(batch), target)

    def training_step(self, batch, batch_index):
        loss = self._loss(batch)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_index):
        loss = self._loss(batch)
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_index):
        loss = self._loss(batch)
        prediction = self(batch)
        mae = torch.mean(torch.abs(prediction - batch["emean"]))
        self.log("test_loss", loss, on_epoch=True)
        self.log("test_mae", mae, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_index, dataloader_index=0):
        return {
            "prediction": self(batch).detach().cpu(),
            "target": batch["emean"].detach().cpu(),
            "image_name": list(batch["image_name"]),
            "patient_id": list(batch["patient_id"]),
        }

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.regressor.parameters(),
            lr=self.learning_rate,
            weight_decay=1e-4,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=6,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }

In [ ]:
def collect_translation(outputs):
    gray = torch.cat([output["gray"] for output in outputs])
    predictions = torch.cat([output["prediction"] for output in outputs])
    targets = torch.cat([output["target"] for output in outputs])
    image_names = [name for output in outputs for name in output["image_name"]]
    patient_ids = [name for output in outputs for name in output["patient_id"]]
    return gray, predictions, targets, image_names, patient_ids


def evaluate_translation(model, trainer, loader, cohort_name, output_prefix):
    outputs = trainer.predict(model, loader)
    gray, predictions, targets, image_names, patient_ids = collect_translation(outputs)
    metric_frame = image_metric_frame(predictions, targets, image_names, patient_ids)
    summary, patient_frame = summarize_image_metrics(metric_frame)
    summary["fid"] = calculate_fid(
        predictions,
        targets,
        NOTEBOOK_DIR / f"fid_{output_prefix}",
        "cuda:0" if torch.cuda.is_available() else "cpu",
    )
    metric_frame.to_csv(NOTEBOOK_DIR / f"{output_prefix}_image_metrics.csv", index=False)
    patient_frame.to_csv(NOTEBOOK_DIR / f"{output_prefix}_patient_image_metrics.csv", index=False)
    return summary, gray, predictions, targets


def collect_regression(outputs):
    predictions = torch.cat([output["prediction"] for output in outputs]).numpy()
    targets = torch.cat([output["target"] for output in outputs]).numpy()
    image_names = [name for output in outputs for name in output["image_name"]]
    patient_ids = [name for output in outputs for name in output["patient_id"]]
    return predictions, targets, image_names, patient_ids


def evaluate_regression(model, trainer, loader, output_prefix):
    outputs = trainer.predict(model, loader)
    predictions, targets, image_names, patient_ids = collect_regression(outputs)
    image_frame, patient_frame = regression_frames(
        predictions,
        targets,
        image_names,
        patient_ids,
    )
    image_frame.to_csv(NOTEBOOK_DIR / f"{output_prefix}_predictions.csv", index=False)
    patient_frame.to_csv(
        NOTEBOOK_DIR / f"{output_prefix}_patient_predictions.csv",
        index=False,
    )
    return summarize_regression(image_frame, patient_frame)

In [ ]:
translation_model = TranslationLightning(
    use_fem=USE_FEM,
    learning_rate=LEARNING_RATE_GENERATOR,
)
if training and USE_PRETRAINED:
    initialize_map2sat(translation_model.network.generator, PRETRAINED_GENERATOR)
generator_best = BestStateCallback("val_reconstruction", "network")
generator_trainer = pl.Trainer(
    max_epochs=GENERATOR_EPOCHS,
    accelerator="auto",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    deterministic=True,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
    enable_progress_bar=False,
    callbacks=[
        EpochRecorder(NOTEBOOK_DIR / f"{EXPERIMENT_NAME}_generator_epoch_metrics.csv"),
        generator_best,
        pl.callbacks.EarlyStopping(
            monitor="val_reconstruction",
            mode="min",
            patience=15,
        ),
    ],
    num_sanity_val_steps=0,
)
if training:
    generator_trainer.fit(translation_model, train_loader, validation_loader)
    save_weight_only(translation_model.network, GENERATOR_WEIGHT)
    reloaded_network = TranslationNetwork(use_fem=USE_FEM)
    load_weight_only(reloaded_network, GENERATOR_WEIGHT)
    translation_model.network.load_state_dict(reloaded_network.state_dict())
else:
    if not GENERATOR_WEIGHT.is_file():
        raise FileNotFoundError(GENERATOR_WEIGHT)
    load_weight_only(translation_model.network, GENERATOR_WEIGHT)
generator_trainer.validate(translation_model, validation_loader)
generator_trainer.test(translation_model, test_loader)
internal_image_summary, gray_examples, prediction_examples, target_examples = evaluate_translation(
    translation_model,
    generator_trainer,
    test_loader,
    "internal_test",
    f"{EXPERIMENT_NAME}_internal_test",
)
image_metrics = {"internal_test": internal_image_summary}
if external_loader is not None:
    generator_trainer.test(translation_model, external_loader)
    external_image_summary, external_gray, external_prediction, external_target = evaluate_translation(
        translation_model,
        generator_trainer,
        external_loader,
        "external_test",
        f"{EXPERIMENT_NAME}_external_test",
    )
    image_metrics["external_test"] = external_image_summary
if EXPORT_ALL_SPLITS:
    save_translation_grid(
        gray_examples,
        prediction_examples,
        target_examples,
        NOTEBOOK_DIR / f"{EXPERIMENT_NAME}_internal_examples.png",
    )
    if external_loader is not None:
        save_translation_grid(
            external_gray,
            external_prediction,
            external_target,
            NOTEBOOK_DIR / f"{EXPERIMENT_NAME}_external_examples.png",
        )

In [ ]:
log_targets = np.log1p(train_frame["emean"].to_numpy(dtype=float))
target_mean = float(log_targets.mean())
target_std = float(log_targets.std(ddof=1))
if not np.isfinite(target_std) or target_std <= 0:
    raise ValueError("Invalid Emean standard deviation")
emean_model = EmeanRegressionLightning(
    input_mode="virtual_swe",
    target_mean=target_mean,
    target_std=target_std,
    translation=translation_model.network,
    learning_rate=LEARNING_RATE_EMEAN,
)
emean_best = BestStateCallback("val_loss", "regressor")
emean_trainer = pl.Trainer(
    max_epochs=EMEAN_EPOCHS,
    accelerator="auto",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    deterministic=True,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
    enable_progress_bar=False,
    callbacks=[
        EpochRecorder(NOTEBOOK_DIR / f"{EXPERIMENT_NAME}_emean_epoch_metrics.csv"),
        emean_best,
        pl.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=12,
        ),
    ],
    num_sanity_val_steps=0,
)
if training:
    emean_trainer.fit(emean_model, train_emean_loader, validation_evaluation_loader)
    save_weight_only(emean_model.regressor, EMEAN_WEIGHT)
    reloaded_regressor = EmeanRegressor()
    load_weight_only(reloaded_regressor, EMEAN_WEIGHT)
    emean_model.regressor.load_state_dict(reloaded_regressor.state_dict())
else:
    if not EMEAN_WEIGHT.is_file():
        raise FileNotFoundError(EMEAN_WEIGHT)
    load_weight_only(emean_model.regressor, EMEAN_WEIGHT)
emean_trainer.validate(emean_model, validation_evaluation_loader)
emean_trainer.test(emean_model, test_evaluation_loader)
emean_metrics = {
    "internal_test": evaluate_regression(
        emean_model,
        emean_trainer,
        test_evaluation_loader,
        f"{EXPERIMENT_NAME}_internal_test",
    )
}
if EXPORT_ALL_SPLITS:
    emean_metrics["train"] = evaluate_regression(
        emean_model,
        emean_trainer,
        train_evaluation_loader,
        f"{EXPERIMENT_NAME}_train",
    )
    emean_metrics["validation"] = evaluate_regression(
        emean_model,
        emean_trainer,
        validation_evaluation_loader,
        f"{EXPERIMENT_NAME}_validation",
    )
if external_evaluation_loader is not None:
    emean_trainer.test(emean_model, external_evaluation_loader)
    emean_metrics["external_test"] = evaluate_regression(
        emean_model,
        emean_trainer,
        external_evaluation_loader,
        f"{EXPERIMENT_NAME}_external_test",
    )
dump_json(
    {"dataset": DATASET_NAME, "experiment": EXPERIMENT_NAME, "image_generation": image_metrics, "emean": emean_metrics},
    METRICS_PATH,
)